# Sub-Millisecond IBM Z Real-Time AI Triage & Emergency Response System
## Phase 1: Clinical Data Pipeline, MIMIC-IV Telemetry & NEWS2 Scoring
**IBM Z Datathon 2026** — *Track: Real-Time AI for Critical Decisions*
**Platform**: IBM LinuxONE (s390x Architecture with Telum AI Acceleration)
---
This notebook establishes the physiological telemetry ingestion pipeline, validates Royal College of Physicians **NEWS2 (National Early Warning Score 2)** calculations, and explores synthetic MIMIC-IV streaming biometrics for emergency medical dispatch.

### 1. Hardware Architecture Verification (IBM LinuxONE s390x)
Verify native execution on IBM Z mainframe infrastructure.

In [ ]:
import platform, os, sys
print('Machine Architecture :', platform.machine())
print('Operating System     :', platform.system(), platform.release())
print('Python Version       :', sys.version.split()[0])
print('Byte Order           :', sys.byteorder.upper(), 'Endian')


### 2. Import Libraries & Clinical NEWS2 Engine

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
sys.path.append('..')
sys.path.append('.')

from pipeline.news2_engine import calculate_news2
from pipeline.vitals_generator import generate_single_sample
from pipeline.stream_manager import PatientStreamBuffer
from pipeline.dispatch_generator import AmbulanceDispatchUnit

print('Clinical triage modules loaded successfully.')


### 3. Load & Inspect MIMIC-IV Clinical Vitals Dataset
Load the 10,000 balanced patient records generated for training.

In [ ]:
data_path = Path('../data/mimic_vitals_train.csv')
if not data_path.exists():
    data_path = Path('data/mimic_vitals_train.csv')

df = pd.read_csv(data_path)
print('Dataset Shape:', df.shape, f'(Total records: {len(df):,})')
df.head(10)


### 4. Statistical Distribution of Clinical Parameters

In [ ]:
display_cols = ['heart_rate', 'spo2', 'systolic_bp', 'respiration_rate', 'temperature', 'news2_score']
df[display_cols].describe().round(2)


### 5. NEWS2 Risk Stratification & ICU Emergency Triggers
Examine the distribution of clinical risk levels (Low, Medium, High/ICU Alert).

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
palette = {'LOW': '#2ecc71', 'MEDIUM': '#f39c12', 'HIGH': '#e74c3c'}
sns.countplot(data=df, x='risk_level', order=['LOW', 'MEDIUM', 'HIGH'], palette=palette)
plt.title('Distribution of Patient Risk Levels (NEWS2)')
plt.xlabel('Clinical Risk Tier')
plt.ylabel('Count')

plt.subplot(1, 2, 2)
sns.histplot(data=df, x='news2_score', hue='risk_level', palette=palette, discrete=True, kde=False)
plt.axvline(7, color='red', linestyle='--', label='ICU Trigger Threshold (>=7)')
plt.title('NEWS2 Score Histogram & Critical Boundary')
plt.xlabel('Total NEWS2 Score')
plt.legend()

plt.tight_layout()
plt.show()


### 6. Correlation Heatmap: Biometrics vs Deterioration Risk

In [ ]:
num_cols = ['heart_rate', 'spo2', 'systolic_bp', 'respiration_rate', 'temperature', 'news2_score']
corr = df[num_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', cbar=True, vmin=-1, vmax=1)
plt.title('Biometric Feature Correlations with NEWS2 Severity')
plt.tight_layout()
plt.show()


### 7. Simulation: Real-Time Ambulance Transit & In-Memory 30s Buffer
Demonstrates real-time telemetry processing and rapid slope-based deterioration detection.

In [ ]:
import time
buffer = PatientStreamBuffer(patient_id='AMB-UNIT-42', max_window_seconds=30.0)
ambulance = AmbulanceDispatchUnit(unit_id='AMB-UNIT-42')

print('Simulating 10 consecutive telemetry frames (Acute Decompensation):')
now = time.time()
for step in range(10):
    reading = {
        'timestamp': now + step * 2.0,
        'heart_rate': 80 + step * 5.0,
        'spo2': 98 - step * 1.5,
        'systolic_bp': 120 - step * 3.0,
        'respiration_rate': 16 + step * 1.5,
        'temperature': 37.0 + step * 0.1,
        'news2_score': min(18, step * 2)
    }
    features = buffer.add_reading(reading)
    amb_state = ambulance.step(dt_seconds=2.0, high_priority=features['news2_max_30s'] >= 7)
    
    if step in [0, 4, 9]:
        print(f"Step {step:2d} | HR: {reading['heart_rate']:5.1f} | SpO2: {reading['spo2']:4.1f}% | SBP: {reading['systolic_bp']:5.1f} | NEWS2: {reading['news2_score']:2d} | Acute Warning: {features['acute_decompensation_warning']} | Siren: {amb_state['siren_active']} | ETA: {amb_state['eta_minutes']:4.1f} min")


### 8. Phase 1 Completion Summary
All Phase 1 requirements are satisfied:
1. Standard Royal College of Physicians NEWS2 clinical scoring engine verified.
2. 10,000 training and 2,500 validation patient samples exported.
3. Real-time streaming buffer and fleet dispatch simulator operational.
4. Ready for **Phase 2: XGBoost / ONNX Model Training & Sub-millisecond IBM Z Benchmarking**.